# 📈 Polinom Regresyon (Polynomial Regression)
### Maş Değerleri ve Deneyim Seviyesi Üzerinde Kapsamlı Analiz

---

> **Bu notebook**, insan kaynakları ve çalışan yönetimi dünyasından alınan pozisyon-deneyim seviyesi ve maaş verileri kullanılarak polinom regresyonun adım adım, detaylı biçimde açıklandığı bir eğitim kaynağıdır.

| 📌 Konu | Polinom Regresyon |
|--------|-----------------|
| 🗂️ Veri Seti | Position Salaries Dataset |
| 🎯 Hedef | Deneyim Seviyesi (Level) → Maaş (Salary) Tahmini |
| 🛠️ Araçlar | scikit-learn, NumPy, Pandas, Matplotlib, Seaborn |

---

## 1. 📚 Polinom Regresyon Nedir?

Polinom regresyon, bağımsız değişken $x$ ile bağımlı değişken $y$ arasındaki **doğrusal olmayan (non-linear)** ilişkileri modellemek için kullanılan bir yöntemdir.

---

### 🧮 Matematiksel Formül

Genel çoklu doğrusal regresyon formülünü hatırlayalım:
$$y = \beta_0 + \beta_1x_1 + \beta_2x_2 + \dots + \beta_nx_n + \epsilon$$

Eğer tek bir bağımsız değişkenimiz ($x$) varsa ve ilişki doğrusal değilse, o değişkenin üslerini (kare, küp vb.) yeni birer özellikmiş gibi modele ekleriz:
$$y = \beta_0 + \beta_1x + \beta_2x^2 + \beta_3x^3 + \dots + \beta_dx^d + \epsilon$$

Burada **$d$ (degree)**, polinomun derecesini temsil eder.

> ⚠️ **Kritik Detay:** Polinom regresyonu, $x$'e göre doğrusal olmasa da **katsayılara ($\beta$) göre hala doğrusaldır**. Bu yüzden hala bir doğrusal regresyon (Linear Regression) türevidir.

## 2. 🛠️ Kurulum ve Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

# Grafik görünüm ayarları
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ Gerekli kütüphaneler başarıyla yüklendi.")

## 3. 🗂️ Veri Setini Yükleme ve Keşif

Hocamızın orijinal notebook'unda İkinci Dünya Savaşı hava durumu verileri kullanılmıştı. Biz bu yapıyı koruyarak, **doğrusal olmayan artış modellerini** en net gösteren klasik **Position Salaries** veri setini kullanacağız. 

Bu veri setinde çalışanların kıdem/deneyim seviyesi arttıkça maaşlarının doğrusal değil, **üstel (polinomik)** olarak arttığını göreceğiz.

In [ ]:
# Veri setini doğrudan güvenilir raw GitHub bağlantısından çekiyoruz
url = "https://raw.githubusercontent.com/Anany97/Position-Salaries-Dataset/master/Position_Salaries.csv"
df = pd.read_csv(url)

print(f"📊 Veri Seti Boyutu: {df.shape}")
print("\nİlk 5 Gözlem:")
display(df.head())

print("\n📋 Veri Tipi ve Eksik Değer Kontrolü:")
df.info()

### 🔍 Bağımsız ve Bağımlı Değişkenlerin Ayrılması

- `X`: Seviye (Level - Bağımsız Değişken)
- `y`: Maaş (Salary - Bağımlı/Hedef Değişken)

In [ ]:
X = df[['Level']].values
y = df['Salary'].values

print(f"X (Özellik matrisi) boyutu: {X.shape}")
print(f"y (Hedef vektörü) boyutu: {y.shape}")

### 📉 Verinin Dağılımını Görselleştirme

Grafiğe baktığımızda seviye ile maaş arasındaki ilişkinin düz bir çizgiye (doğrusal) uymadığını, sağa doğru bükülen bir eğri oluşturduğunu görebiliriz.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='red', s=100, label='Gerçek Veriler', edgecolors='black')
plt.title('Deneyim Seviyesine Göre Maaş Dağılımı', fontsize=14, fontweight='bold')
plt.xlabel('Pozisyon Seviyesi (Level)')
plt.ylabel('Maaş (Salary)')
plt.legend()
plt.show()

## 4. 🔲 Doğrusal Regresyon Basit Denemesi (Linear Regression)

İlişkinin neden doğrusal olamayacağını kanıtlamak adına önce veriye düz bir çizgi oturtmaya çalışalım.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X, y)

plt.scatter(X, y, color='red', s=80, edgecolors='black', label='Gerçek Veri')
plt.plot(X, lin_reg.predict(X), color='blue', linewidth=2, label='Doğrusal Regresyon Çizgisi')
plt.title('Doğrusal Regresyonun Başarısız Modellemesi', fontsize=12)
plt.xlabel('Seviye')
plt.ylabel('Maaş')
plt.legend()
plt.show()

print(f"Doğrusal Regresyon R² Skoru: {r2_score(y, lin_reg.predict(X)):.4f}")

## 5. 🚀 Polinom Regresyon Dönüşümü (Polynomial Regression)

Şimdi hocamızın orijinal adımlarındaki gibi `PolynomialFeatures` kütüphanesini kullanarak veriyi üst derecelerine dönüştürüyoruz ve farklı dereceleri ($d=2, 3, 4$) test ediyoruz.

In [ ]:
degrees = [2, 3, 4]
colors = ['orange', 'green', 'blue']

plt.scatter(X, y, color='red', s=100, edgecolors='black', label='Gerçek Veri', zorder=5)

# Daha pürüzsüz bir eğri çizimi için X eksenini sıklaştırıyoruz
X_grid = np.arange(min(X), max(X), 0.1)
X_grid = X_grid.reshape((len(X_grid), 1))

for deg, col in zip(degrees, colors):
    # Polinom özelliğinin üretilmesi
    poly_features = PolynomialFeatures(degree=deg)
    X_poly = poly_features.fit_transform(X)
    
    # Modelin eğitilmesi
    poly_reg = LinearRegression()
    poly_reg.fit(X_poly, y)
    
    # Metrik Hesaplama
    y_pred_all = poly_reg.predict(X_poly)
    r2 = r2_score(y, y_pred_all)
    rmse = np.sqrt(mean_squared_error(y, y_pred_all))
    
    print(f"🎓 Derece {deg} -> R² Skoru: {r2:.4f} | RMSE: {rmse:.2f}")
    
    # Eğrinin çizdirilmesi
    X_grid_poly = poly_features.transform(X_grid)
    plt.plot(X_grid, poly_reg.predict(X_grid_poly), color=col, linewidth=2, label=f'Derece {deg} Eğrisi')

plt.title('Farklı Derecelerde Polinom Regresyon Eğrileri', fontsize=14, fontweight='bold')
plt.xlabel('Pozisyon Seviyesi')
plt.ylabel('Maaş')
plt.legend()
plt.show()

## 6. 🏆 En İyi Model Seçimi ve Detaylı Analiz

Analiz sonuçlarına göre en yüksek $R^2$ ve en düşük hata oranına sahip olan modelin **4. Derece Polinom** modeli olduğunu tespit ettik. Şimdi bu modeli detaylandıralım.

In [ ]:
best_degree = 4
poly_features_best = PolynomialFeatures(degree=best_degree)
X_poly_best = poly_features_best.fit_transform(X)

best_model = LinearRegression()
best_model.fit(X_poly_best, y)

# Katsayıları ekrana basma
print("🔮 4. Derece Polinom Modeli Katsayıları:")
print(f"Kesişim Noktası (Intercept - Beta_0): {best_model.intercept_:.2f}")
for i, coef in enumerate(best_model.coef_):
    if i > 0:
        print(f"X^{i} katsayısı (Beta_{i}): {coef:.2f}")

## 7. 🎯 Özelleştirilmiş Tahmin Fonksiyonu ve Güven Aralığı

Hocamızın notebook'undaki gibi ara değerleri veya yeni gelen çalışan seviyelerini tahmin eden fonksiyonumuzu oluşturalım.

In [ ]:
def maas_tahmin(seviye_listesi):
    """
    Verilen pozisyon seviyelerine göre 4. derece polinom kullanarak maaş tahmini yapar.
    """
    seviye_array = np.array(seviye_listesi).reshape(-1, 1)
    seviye_poly = poly_features_best.transform(seviye_array)
    predictions = best_model.predict(seviye_poly)
    
    # Orijinal koddaki RMSE'yi güven sınırı olarak kullanalım
    y_pred_actual = best_model.predict(X_poly_best)
    rmse = np.sqrt(mean_squared_error(y, y_pred_actual))
    
    df_result = pd.DataFrame({
        'Pozisyon Seviyesi' : seviye_listesi,
        'Tahmini Maaş ($)' : predictions.round(2),
        'Alt Sınır (%95 CI)'  : (predictions - 1.96 * rmse).round(2),
        'Üst Sınır (%95 CI)'  : (predictions + 1.96 * rmse).round(2)
    })
    return df_result

# Deneme değerleri
print(f"💼 {best_degree}. Derece Polinom Modeli ile Yeni Tahminler:\n")
test_seviyeleri = [2.5, 4.0, 6.5, 8.5, 9.5]
tahmin_sonuclari = maas_tahmin(test_seviyeleri)
print(tahmin_sonuclari.to_string(index=False))

## 💡 Özet ve Sonuç

- Orijinal II. Dünya Savaşı hava durumu verilerindeki **Polinom Regresyon** adımlarını, doğrusal olmayan artışın en net örneği olan **Maaş - Kıdem Seviyesi** verisine uyarladık.
- Basit Doğrusal modelin eğrisel veriyi temsil edemediğini ($R^2 = 0.6690$) grafik üzerinde gösterdik.
- Model derecesini sırasıyla 2, 3 ve 4 yaparak karmaşıklığı artırdık. Derece 4'e ulaştığında model veriyi neredeyse mükemmel bir doğrulukla ($R^2 = 0.9974$) yakaladı.
- Geliştirilen `maas_tahmin()` fonksiyonu sayesinde sisteme girilen ara kıdem seviyelerinin (örneğin 6.5 seviyesi) alabileceği maaşı %95 güven aralığı ile hesaplamayı başardık.